#### This shall be run upon generating some example .pkl files in the directory defined below. This is done with main.py

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import math
import numpy as np
import utm
import matplotlib.pyplot as plt
from shapely.geometry import LineString, Point, Polygon, box
from shapely.ops import nearest_points
import pickle
from osm_wrapper import OSMWrapper
from geopy import Point as GeoPoint
from utils.enums import Direction, MapObjectId


osm_data_path = "debug_output/test_osm"
osm_data = os.listdir(osm_data_path)

out_figures_path = "debug_output/figures"


In [8]:
from utils.transformations import translate, rotate, transform_to_vehicle_coordinates
wrapper = OSMWrapper()
            
def get_links_around_egovehicle(ego_vehicle_latlon, wrapper=wrapper) -> list:
        """Retrieves the links around the vehicle location."""
        origin_x, origin_y, utm_zone_num, utm_zone_letter = utm.from_latlon(ego_vehicle_latlon[0], ego_vehicle_latlon[1])
        origin = Point(origin_x, origin_y)
        
        origin_latlon = utm.to_latlon(origin.x, origin.y, utm_zone_num, utm_zone_letter)
        geopoint_origin = GeoPoint(origin_latlon[0], origin_latlon[1])

        links_in_enlarged_area = wrapper.get_links(geopoint_origin, 1000)
        link_ids = [link.get_ID() for link in links_in_enlarged_area]

        return link_ids
    

def link_id_object(string_id):
    return MapObjectId(int(string_id.split("-")[0]), int(string_id.split("-")[1]))

def plot_links(inputs, ego_vehicle_latlon, ego_heading, wrapper, ax):
    """Takes links and returns the corresponding labelled lane information."""
    
    ego_vehicle_data = {
        "ego_vehicle_lat": ego_vehicle_latlon[0],
        "ego_vehicle_lon": ego_vehicle_latlon[1],
        "ego_vehicle_yaw": ego_heading,
    }
    
    links = []
    for link_id in inputs:
        if link_id:
            sd_object = wrapper.get_sd_object_by_id(link_id_object(link_id))
            if sd_object:  # Check if the list is not empty
                links.append(sd_object[0])

    for idx, link in enumerate(links):
        vectorized_points = transform_to_vehicle_coordinates(ego_vehicle_data, link)

        line = LineString(vectorized_points)
        if len(line.coords) > 1:
            color, lw, zorder, alpha = ('lightgrey', 1, 99, 1) #if not route_flag else ('r', 1, 100, 1)
            ax.plot(*line.coords.xy, c=color, lw=lw, zorder=zorder, alpha=alpha)

    

In [ ]:
# rows = 3
# osm_data = osm_data[:rows]

# Function to create, display, and save figures one at a time
def create_and_save_figures(osm_data):
    os.makedirs(out_figures_path, exist_ok=True)

    for i, sample in enumerate(osm_data):
        fig, ax = plt.subplots(1, 1, figsize=(6, 4))  # Single subplot since MM data is removed
        input_file_path = os.path.join(osm_data_path, sample)
        input_data = pickle.load(open(input_file_path, "rb"))
        
        gt = np.array([input_data["gt"]["local_lat"], input_data["gt"]["local_lon"]]).T
        
        route = np.array([a for a in input_data["route_coords"]])
        ego_vehicle_latlon = (input_data["pred_time"]["lat"], input_data["pred_time"]["lon"])
        ego_heading = input_data["pred_time"]["heading"]
        lanes = get_links_around_egovehicle(ego_vehicle_latlon, wrapper)
        
        # Plot OSM data
        plot_links(lanes, ego_vehicle_latlon, ego_heading, wrapper, ax)
        ax.plot(route[:, 0], route[:, 1], label='Route', color='red')
        ax.plot(gt[:, 0], gt[:, 1], label='GT', color='green')
        ax.set_aspect('equal', adjustable='box')
        ax.set_xlabel('Y')
        ax.set_ylabel('X')
        ax.set_xlim(-150, 150)
        ax.set_ylim(-20, 175)
        
        # Add a title with the lat/lon info
        lat = input_data["pred_time"]["lat"]
        lon = input_data["pred_time"]["lon"]
        namestring = f"({lat:.4f}, {lon:.4f})"
        fig.suptitle(namestring, fontsize=16)
        
        # Save and display the figure
        fig.tight_layout()
        fig.savefig(os.path.join(out_figures_path, f'sample_{i+1}_{namestring}.png'))
        plt.show()  # Display the figure
        
        # Close the figure to free memory
        plt.close(fig)

# Create and save figures one at a time
print("Positive x points towards the right of the vehicle")
create_and_save_figures(osm_data)


### OSMNX package experiments

In [ ]:
import osmnx as ox
import matplotlib.pyplot as plt

# Parameters
lat = 53.8415575
lon = 18.6265579
dist = 500
network_type = 'drive_service'
retail_all = True
truncate_by_edge = True
simplify = False

# Get the graph
graph = ox.graph_from_point(
    (lat, lon), dist=dist, network_type=network_type,
    retain_all=retail_all, truncate_by_edge=truncate_by_edge, simplify=simplify
)

# Plot the graph
fig, ax = ox.plot_graph(graph, show=False, close=False, edge_color='white', edge_linewidth=0.5, node_size=5)

# Show figure dimensions
fig_width, fig_height = fig.get_size_inches()
x_limits = ax.get_xlim()
y_limits = ax.get_ylim()

print(f"Figure size (width x height in inches): {fig_width} x {fig_height}")
print(f"X-axis limits (Longitude): {x_limits}")
print(f"Y-axis limits (Latitude): {y_limits}")

fig.show()


In [ ]:
import osmnx as ox
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from math import cos, radians

# Parameters
lat = 53.8415575
lon = 18.6265579
dist = 500  # Distance in meters
network_type = 'drive_service'
retain_all = True
truncate_by_edge = False
simplify = False

# Get the graph
graph = ox.graph_from_point(
    (lat, lon), dist=dist, network_type=network_type,
    retain_all=retain_all, truncate_by_edge=truncate_by_edge, simplify=simplify
)

# Plot the graph
fig, ax = ox.plot_graph(graph, show=False, close=False, edge_color='white', edge_linewidth=0.5, node_size=5)

# Convert distance to degrees
# Latitude degrees (constant)
dist_deg_lat = dist / 111139  # 1 degree latitude ~ 111.139 km

# Longitude degrees (adjust for latitude)
dist_deg_lon = dist / (111139 * cos(radians(lat)))

# Add a circle to the plot
circle = Circle(
    (lon, lat), dist_deg_lat,  # Use latitude for circular overlay
    edgecolor='red', facecolor='none', linestyle='--', linewidth=1
)
ax.add_patch(circle)

# Adjust the limits to include the circle if necessary
ax.set_xlim(min(lon - dist_deg_lon, ax.get_xlim()[0]), max(lon + dist_deg_lon, ax.get_xlim()[1]))
ax.set_ylim(min(lat - dist_deg_lat, ax.get_ylim()[0]), max(lat + dist_deg_lat, ax.get_ylim()[1]))

# Show figure dimensions
fig_width, fig_height = fig.get_size_inches()
x_limits = ax.get_xlim()
y_limits = ax.get_ylim()

print(f"Figure size (width x height in inches): {fig_width} x {fig_height}")
print(f"X-axis limits (Longitude): {x_limits}")
print(f"Y-axis limits (Latitude): {y_limits}")

# Show the final plot with the circle
plt.show()


In [ ]:
import osmnx as ox
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle
import utm

# Parameters
lat = 53.8415575
lon = 18.6265579
dist = 700  # Distance in meters
network_type = 'drive_service'
retain_all = True
truncate_by_edge = True
simplify = False

# Get the graph and project it to UTM (meters)
graph = ox.graph_from_point(
    (lat, lon), dist=dist, network_type=network_type,
    retain_all=retain_all, truncate_by_edge=truncate_by_edge, simplify=simplify
)
graph_proj = ox.project_graph(graph)

# Convert latitude and longitude to UTM coordinates
utm_coords = utm.from_latlon(lat, lon)

# Get the UTM coordinates
center_x, center_y = utm_coords[0], utm_coords[1]

# Plot the graph
fig, ax = ox.plot_graph(graph_proj, show=False, close=False, edge_color='white', edge_linewidth=0.5, node_size=5)

# Add a circle in meters
circle = Circle(
    (center_x, center_y), dist,
    edgecolor='red', facecolor='none', linestyle='--', linewidth=1
)
ax.add_patch(circle)

# Add a square (2x distance) around the circle
square_side = dist * 2  # Side length in meters
square = Rectangle(
    (center_x - square_side / 2, center_y - square_side / 2),
    square_side, square_side,
    edgecolor='orange', facecolor='none', linestyle='--', linewidth=1
)
ax.add_patch(square)

# Ensure the circle and square fit in the plot
# ax.set_xlim(center_x - square_side / 2, center_x + square_side / 2)
# ax.set_ylim(center_y - square_side / 2, center_y + square_side / 2)

# Show the final plot
plt.show()
